# Track NN на хвосте скора

Что track-модель предсказывает для с высоким score exp-событий: направление,
точку на линии, сырые `sigma` и число хитов/строк. BARS fit не используется.

Входы: classifier exp-DB (`exp_full_thr0p8.duckdb`), track-NN exp-DB и
10% test-MC (`muatm`/`nuatm`/`nue2`) как референс. Track-DB содержит только
события, прошедшие `p >= 0.70`, >=8 хитов / >=2 строки. `NOT_EXCLUDED`
исключает два известных плохих c02-рана.

`track_sigma_*` — сырые, некалиброванные; для exp/канонического SN
калиброванные радиусы NULL. Это не coverage.


In [ ]:
CHECKPOINT = ("261024_1202_da_nu_classifier_exp_full_E1_weights_z128_k10_"
              "dlr1e-2_lam0p003_delay2_h8s3_muOnlyDA_s42@best_da_model")
TRACK_DB_EXP = ("/home/albert/Baikal2025/inference/angle_track_uncertainty/preds/"
                "from_nu_261024_1202_muOnlyDA_h8s3/release_v3@angle_track_uncertainty/"
                "exp_full_canonical_p070_track.duckdb")
TRACK_DB_MC = ("/home/albert/Baikal2025/inference/angle_track_uncertainty/preds/"
               "from_nu_261024_1202_muOnlyDA_h8s3_mc/release_v3@angle_track_uncertainty/"
               "mc_merged_canonical_p070_track.duckdb")
SCORE_CUTS = [0.5, 0.7, 0.8, 0.9, 0.95]
MC_MODULUS = 10


In [ ]:
import os, sys
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")
ROOT = "/home/albert/Baikal2025"
sys.path.insert(0, ROOT)
sys.path.insert(0, ROOT + "/inference/nu_classifier/analysis/HIGH_SCORE_EXCESS")
os.chdir(ROOT + "/inference/nu_classifier/analysis/HIGH_SCORE_EXCESS")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

from excess.tracknn import load_events, load_scored_mc

EV, report = load_events(
    CHECKPOINT, TRACK_DB_EXP, TRACK_DB_MC, mc_modulus=MC_MODULUS)
exp, muatm, nuatm, nue2 = EV["exp_full"], EV["muatm"], EV["nuatm"], EV["nue2"]

# Полный (не прореженный) muatm выше min(SCORE_CUTS): в 10% выборке
# high-score хвост слишком мал для сравнения соответствующих групп.
muatm_scored = load_scored_mc(
    CHECKPOINT, TRACK_DB_MC, data_class="muatm_2020",
    score_floor=min(SCORE_CUTS))

report


In [ ]:
# Sanity: полный join, отсутствие fit-колонок, чистая exp-популяция.
assert not any(c.startswith("fit_") for c in exp.columns)
assert exp["track_theta_deg"].notna().all(), "exp events without track prediction"
for name, frame in [("muatm", muatm), ("nuatm", nuatm), ("nue2", nue2),
                    ("muatm_scored", muatm_scored)]:
    assert frame["track_theta_deg"].notna().all(), name
assert int(exp["bad_run"].sum()) == 0, "bad exp run leaked through NOT_EXCLUDED"

sanity = pd.DataFrame({
    "домен": list(EV) + ["muatm_scored"],
    "событий": [len(EV[k]) for k in EV] + [len(muatm_scored)],
    "track_matched": ([int(EV[k]["track_theta_deg"].notna().sum()) for k in EV]
                      + [int(muatm_scored["track_theta_deg"].notna().sum())]),
    "score_max": ([float(EV[k]["score"].max()) for k in EV]
                  + [float(muatm_scored["score"].max())]),
})
sanity


In [ ]:
rows = []
for cut in SCORE_CUTS:
    e = exp.loc[exp["score"] >= cut]
    m = muatm_scored.loc[muatm_scored["score"] >= cut]
    rows.append({
        "score >=": cut,
        "n_exp": len(e),
        "n_muatm": len(m),
        "exp_theta_med": e["track_theta_deg"].median(),
        "muatm_theta_med": m["track_theta_deg"].median(),
        "exp_frac<90": float((e["track_theta_deg"] < 90).mean()) if len(e) else np.nan,
        "muatm_frac<90": float((m["track_theta_deg"] < 90).mean()) if len(m) else np.nan,
        "exp_sigma_med": e["track_sigma_angle_deg"].median(),
        "muatm_sigma_med": m["track_sigma_angle_deg"].median(),
    })
counts = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(exp["score"], bins=100, range=(0, 1), log=True, histtype="step",
        label="exp")
ax.hist(muatm_scored["score"], bins=100, range=(0, 1), log=True,
        histtype="step", label="muatm (full, score>=%.2f)" % min(SCORE_CUTS))
for cut in SCORE_CUTS:
    ax.axvline(cut, color="0.6", linestyle=":", linewidth=1)
ax.set_xlabel("classifier score")
ax.set_ylabel("events / bin")
ax.set_title("exp и muatm: скор, с отметками SCORE_CUTS")
ax.legend(fontsize=8)
plt.tight_layout()
counts.round(3)


In [ ]:
bins = np.linspace(0, 180, 73)
groups = [
    ("score >= 0.9", exp["score"] >= 0.9, muatm_scored["score"] >= 0.9),
    ("0.8 <= score < 0.9",
     (exp["score"] >= 0.8) & (exp["score"] < 0.9),
     (muatm_scored["score"] >= 0.8) & (muatm_scored["score"] < 0.9)),
    ("0.5 <= score < 0.7",
     (exp["score"] >= 0.5) & (exp["score"] < 0.7),
     (muatm_scored["score"] >= 0.5) & (muatm_scored["score"] < 0.7)),
]
fig, axes = plt.subplots(1, len(groups), figsize=(15, 4), sharey=True)

def hist(ax, values, label, **kwargs):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    ax.hist(values, bins=bins, density=True, histtype="step",
            label=f"{label} (n={len(values)})", **kwargs)

for ax, (label, exp_mask, mu_mask) in zip(axes, groups):
    hist(ax, exp.loc[exp_mask, "track_theta_deg"], "exp", linewidth=2)
    hist(ax, muatm_scored.loc[mu_mask, "track_theta_deg"], "muatm")
    ax.axvline(90, color="k", linestyle=":")
    ax.set_title(label)
    ax.set_xlabel("track NN theta, deg")
    ax.legend(fontsize=8)
axes[0].set_ylabel("density")
fig.suptitle("Одинаковые группы по score: exp vs muatm", y=1.02)
plt.tight_layout()


In [ ]:
d = exp.dropna(subset=["track_theta_deg"])
fig, ax = plt.subplots(figsize=(8, 5))
h = ax.hist2d(d["score"], d["track_theta_deg"], bins=(80, 72),
              range=[[0, 1], [0, 180]], norm=LogNorm(), cmap="viridis")
fig.colorbar(h[3], ax=ax, label="events / bin")

score_bins = np.linspace(0, 1, 21)
centers = 0.5 * (score_bins[:-1] + score_bins[1:])
med = d.groupby(pd.cut(d["score"], score_bins), observed=True)["track_theta_deg"].median()
ax.plot(centers, med.values, "o-", color="white", markersize=3, label="median theta")
ax.axhline(90, color="tab:red", linestyle=":", linewidth=1)
ax.set_xlabel("classifier score")
ax.set_ylabel("track NN theta, deg")
ax.set_title("exp: score vs track theta")
ax.legend(loc="upper right")
plt.tight_layout()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
score_bins = np.linspace(0, 1, 21)

panels = [
    ("track_sigma_angle_deg", "track sigma_angle, deg", True),
    ("track_sigma_anchor_m", "track sigma_anchor, m", True),
    ("track_n_hits", "track n_hits", False),
]
for ax, (column, title, log_y) in zip(axes, panels):
    for frame, name, colour in ((exp, "exp", "C0"),
                                (muatm_scored, "muatm (full, score>=%.2f)"
                                 % min(SCORE_CUTS), "C1")):
        values = frame.dropna(subset=[column])
        curve = values.groupby(
            pd.cut(values["score"], score_bins), observed=True)[column].median()
        # observed=True drops empty bins, so derive x from the curve's own
        # intervals instead of assuming all 20 bins are present.
        mid = np.array([interval.mid for interval in curve.index], dtype=float)
        valid = np.isfinite(curve.values) & np.isfinite(mid)
        ax.plot(mid[valid], curve.values[valid], label=name, color=colour)
    ax.set_xlabel("classifier score")
    ax.set_ylabel(title)
    ax.set_title(title)
    if log_y:
        ax.set_yscale("log")
    ax.legend(fontsize=8)
plt.tight_layout()

pd.DataFrame({
    "score cut": SCORE_CUTS,
    "n_exp": [int((exp["score"] >= c).sum()) for c in SCORE_CUTS],
    "exp_sigma_angle_med": [float(exp.loc[exp["score"] >= c, "track_sigma_angle_deg"].median())
                            for c in SCORE_CUTS],
    "muatm_sigma_angle_med": [float(muatm_scored.loc[muatm_scored["score"] >= c,
                                                     "track_sigma_angle_deg"].median())
                              for c in SCORE_CUTS],
    "exp_sigma_anchor_med": [float(exp.loc[exp["score"] >= c, "track_sigma_anchor_m"].median())
                             for c in SCORE_CUTS],
    "muatm_sigma_anchor_med": [float(muatm_scored.loc[muatm_scored["score"] >= c,
                                                      "track_sigma_anchor_m"].median())
                               for c in SCORE_CUTS],
})


In [ ]:
HIT_EDGES = [0, 8, 10, 12, 15, 20, 30, 50, np.inf]

def hit_bin_stats(frame, mask, label, cut):
    d = frame.loc[mask].dropna(subset=["track_theta_deg"]).copy()
    d["hit_bin"] = pd.cut(d["n_sn_hits"], HIT_EDGES, right=False)
    g = d.groupby("hit_bin", observed=True)
    out = g.agg(
        n=("track_theta_deg", "size"),
        theta_med=("track_theta_deg", "median"),
        theta_q25=("track_theta_deg", lambda s: s.quantile(0.25)),
        theta_q75=("track_theta_deg", lambda s: s.quantile(0.75)),
        sigma_med=("track_sigma_angle_deg", "median"),
        frac_lt90=("track_theta_deg", lambda s: float((s < 90).mean())),
    )
    out.insert(0, "score_cut", cut)
    out.insert(1, "group", label)
    return out.reset_index()

exp_rows, mu_rows = [], []
for cut in SCORE_CUTS:
    exp_rows.append(hit_bin_stats(exp, exp["score"] >= cut, "exp", cut))
    mu_rows.append(hit_bin_stats(muatm_scored, muatm_scored["score"] >= cut,
                                 "muatm", cut))
exp_table = pd.concat(exp_rows, ignore_index=True).drop(columns="group")
mu_table = pd.concat(mu_rows, ignore_index=True).drop(columns="group")
comparison = exp_table.merge(
    mu_table, on=["score_cut", "hit_bin"], suffixes=("_exp", "_muatm"))
comparison["d_theta"] = comparison["theta_med_exp"] - comparison["theta_med_muatm"]
comparison["d_frac_lt90"] = comparison["frac_lt90_exp"] - comparison["frac_lt90_muatm"]
comparison.round(2)


In [ ]:
def angular_error(theta, phi, theta_true, phi_true):
    t1, p1, t2, p2 = np.deg2rad([theta, phi, theta_true, phi_true])
    v1 = np.column_stack([np.sin(t1) * np.cos(p1),
                          np.sin(t1) * np.sin(p1),
                          np.cos(t1)])
    v2 = np.column_stack([np.sin(t2) * np.cos(p2),
                          np.sin(t2) * np.sin(p2),
                          np.cos(t2)])
    return np.rad2deg(np.arccos(np.clip(np.sum(v1 * v2, axis=1), -1.0, 1.0)))

SIGMA_EDGES = [0, 1, 2, 3, 4, 6, 8, 12, 20, 50, 180]
rows = []
for name in ("muatm", "nuatm", "nue2"):
    d = EV[name].dropna(subset=[
        "prime_theta_deg", "prime_phi_deg", "track_theta_deg", "track_phi_deg"])
    d = d.copy()
    d["err_deg"] = angular_error(
        d["track_theta_deg"], d["track_phi_deg"],
        d["prime_theta_deg"], d["prime_phi_deg"])
    d["sigma_bin"] = pd.cut(d["track_sigma_angle_deg"], SIGMA_EDGES, right=False)
    for interval, group in d.groupby("sigma_bin", observed=True):
        if len(group) < 50:
            continue
        rows.append({
            "класс": name,
            "sigma_angle bin": str(interval),
            "sigma_mid": float(interval.mid),
            "n": len(group),
            "err_med_deg": group["err_deg"].median(),
            "err_q68_deg": group["err_deg"].quantile(0.68),
        })
truth_table = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(7, 4))
for name, marker in (("muatm", "o"), ("nuatm", "s"), ("nue2", "^")):
    part = truth_table[truth_table["класс"] == name]
    if len(part):
        ax.plot(part["sigma_mid"], part["err_q68_deg"], marker=marker,
                label=name)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("track sigma_angle, deg (bin midpoint)")
ax.set_ylabel("angular error to truth, q68, deg")
ax.set_title("MC: что реально означает track sigma_angle")
ax.legend()
plt.tight_layout()
truth_table.round(3)


In [ ]:
columns = [
    "event_fk", "part_key", "cluster", "run", "score", "was_da_target",
    "track_theta_deg", "track_phi_deg", "track_sigma_angle_deg",
    "track_sigma_anchor_m", "track_n_hits", "track_n_strings",
    "track_hit_ratio",
]
exp.nlargest(20, "score")[columns]


In [ ]:
by_part = exp.groupby("part_key").agg(
    events=("score", "size"),
    n_high=("score", lambda s: int((s >= 0.9).sum())),
    max_score=("score", "max"),
)
by_part["high_frac"] = by_part["n_high"] / by_part["events"]
by_part.sort_values("n_high", ascending=False).head(30).round(4)


In [ ]:
summary = []
for cut in SCORE_CUTS:
    e = exp.loc[exp["score"] >= cut]
    m = muatm_scored.loc[muatm_scored["score"] >= cut]
    summary.append({
        "score >=": cut,
        "n_exp": len(e),
        "exp_theta_med": e["track_theta_deg"].median(),
        "exp_frac<90": float((e["track_theta_deg"] < 90).mean()),
        "exp_sigma_angle_med": e["track_sigma_angle_deg"].median(),
        "exp_n_hits_med": e["track_n_hits"].median(),
        "n_muatm": len(m),
        "muatm_theta_med": m["track_theta_deg"].median(),
        "muatm_frac<90": float((m["track_theta_deg"] < 90).mean()),
        "muatm_sigma_angle_med": m["track_sigma_angle_deg"].median(),
        "muatm_n_hits_med": m["track_n_hits"].median(),
    })
pd.DataFrame(summary).round(3)


## Как читать

- `muatm_scored` — полный (не прореженный) muatm выше `min(SCORE_CUTS)`;
  `muatm` в `EV` — 10% выборка по партам и используется только для MC truth.
  Абсолютные числа exp и muatm всё равно нельзя сравнивать напрямую: это разные
  экспозиции; сравнивайте медианы, доли и формы внутри одинаковых score-групп.
- `track_sigma_*` для exp — сырые масштабы, не coverage. Для интерпретации
  используйте MC-таблицу `err_deg` vs `sigma_angle`: она показывает, какой
  реальной угловой ошибке соответствует данный `sigma` на MC.
- Track-DB отбирает события при `p >= 0.70 / 8 / 2`, а классификатор — при
  `> 0.8 / 8 / 3`; `track_n_hits` и `n_sn_hits` — разные счётчики.
- Два плохих c02-рана исключены через `reader.NOT_EXCLUDED`; `bad_run` в exp
  должен быть 0.
- MC `muatm` — смесь по числу мюонов, а track-модель обучалась на single-track
  muatm. Truth-таблица на всём `muatm` — ориентир, не чистая шкала; для
  отдельного среза смотрите `prime_response_muons_n`.
